In [ ]:
import tensorflow as tf

try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()  # TPU detection
    print('Running on TPU ', tpu.master())
except ValueError:
    raise BaseException('ERROR: Not connected to a TPU runtime; please see the previous cell in this notebook for instructions!')

tf.config.experimental_connect_to_cluster(tpu)
tf.tpu.experimental.initialize_tpu_system(tpu)
strategy = tf.distribute.TPUStrategy(tpu)  # Use the non-experimental symbol

# Print TPU details
print("TPUClusterResolver:", tpu)
print("Number of replicas:", strategy.num_replicas_in_sync)

Running on TPU  
TPUClusterResolver: <tensorflow.python.distribute.cluster_resolver.tpu.tpu_cluster_resolver.TPUClusterResolver object at 0x791c867039a0>
Number of replicas: 8


In [ ]:
import os
import sys
import numpy as np
from keras import mixed_precision
from keras.models import Model
from keras.layers import Input, Embedding, GRU, Dense, Concatenate, Activation
from keras.optimizers import Adam, RMSprop
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import configuration file for paths
import sys
sys.path.insert(0, '/content/drive/MyDrive/CSCK507_EMA_GroupC_Final')
import config

# Set the paths to the saved training, validation, and test sets
train_data_file = os.path.join(config.MAIN_DIR_PATH, config.TRAIN_DATA)
val_data_file = os.path.join(config.MAIN_DIR_PATH, config.VAL_DATA)
test_data_file = os.path.join(config.MAIN_DIR_PATH, config.TEST_DATA)

Mounted at /content/drive


In [ ]:
# Enable mixed precision training with 'bfloat16' format on TPU
policy = mixed_precision.Policy('mixed_bfloat16')
mixed_precision.set_global_policy(policy)

## **Data Loading**

In [ ]:
# Load the Train Dataset
train_data = np.load(train_data_file)
X_train_enc, X_train_dec, y_train = train_data['X_train_enc'], train_data['X_train_dec'], train_data['y_train']
# Load the Validation Dataset
val_data = np.load(val_data_file)
X_val_enc, X_val_dec, y_val = val_data['X_val_enc'], val_data['X_val_dec'], val_data['y_val']

In [ ]:
# Output shapes for verification
print(f"Train encoder shape: {X_train_enc.shape}, Train decoder shape: {X_train_dec.shape}, Train target shape: {y_train.shape}")
print(f"Validation encoder shape: {X_val_enc.shape}, Validation decoder shape: {X_val_dec.shape}, Validation target shape: {y_val.shape}")

Train encoder shape: (9451799, 30), Train decoder shape: (9451799, 30), Train target shape: (9451799, 30)
Validation encoder shape: (2025386, 30), Validation decoder shape: (2025386, 30), Validation target shape: (2025386, 30)


## **TPU Strategy Initialization**

In [ ]:
# Initialize the TPU Strategy
resolver = tf.distribute.cluster_resolver.TPUClusterResolver()  # Automatically detects the TPU
print(f"TPU Address: {resolver.master()}")  # Print the TPU address for confirmation
strategy = tf.distribute.TPUStrategy(resolver)  # Create TPU strategy for parallel training

# Print detected TPU devices to confirm initialization
print("Logical Devices:", tf.config.list_logical_devices())

TPU Address: 
Logical Devices: [LogicalDevice(name='/device:CPU:0', device_type='CPU'), LogicalDevice(name='/device:TPU:0', device_type='TPU'), LogicalDevice(name='/device:TPU:1', device_type='TPU'), LogicalDevice(name='/device:TPU:2', device_type='TPU'), LogicalDevice(name='/device:TPU:3', device_type='TPU'), LogicalDevice(name='/device:TPU:4', device_type='TPU'), LogicalDevice(name='/device:TPU:5', device_type='TPU'), LogicalDevice(name='/device:TPU:6', device_type='TPU'), LogicalDevice(name='/device:TPU:7', device_type='TPU'), LogicalDevice(name='/device:TPU_SYSTEM:0', device_type='TPU_SYSTEM')]


## **Seq2Seq Model with Attention**

In [ ]:
# Define the Distribution Strategy and Model
with strategy.scope():

    # Define Model Hyper/Parameters
    vocab_size = int(np.max([np.max(X_train_enc), np.max(X_train_dec), np.max(y_train)])) + 1
    embedding_dim = 96
    max_length = X_train_enc.shape[1]
    units = 64

    # MODEL ARCHITECTURE ============================================
    # Encoder
    encoder_inputs = Input(shape=(max_length,), name='Encoder_Input')
    encoder_embedder = Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True, name='encoder_embedding')
    encoder_embedding = encoder_embedder(encoder_inputs)
    encoder_gru = GRU(units, return_sequences=True, return_state=True, name='Encoder_GRU')
    encoder_output, encoder_state = encoder_gru(encoder_embedding)

    # Decoder
    decoder_inputs = Input(shape=(max_length,), name='Decoder_Input')
    decoder_embedder = Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True, name='Decoder_Embedding')
    decoder_embedding = decoder_embedder(decoder_inputs)
    decoder_step_output, decoder_state = GRU(units, return_sequences=True, return_state=True, name='Decoder_GRU')(
        decoder_embedding, initial_state=encoder_state)

    # Apply Luong attention
    # Use decoder_step_output hidden states at each time step
    score = tf.matmul(decoder_step_output, encoder_output, transpose_b=True, name = 'Alignment Scores')
    attention_weights = tf.nn.softmax(score, axis=-1, name = 'Alignment Weights')
    context_vector = tf.matmul(attention_weights, encoder_output, name='Context Vectors')

    # Concatenate context vector with the decoder's GRU output
    decoder_context_concat = Concatenate(axis=-1, name='Attention_Concatenate')([decoder_step_output, context_vector])

    # Dense layer to generate predictions
    decoder_output = Dense(vocab_size, activation='softmax', name='Decoder_Dense')(decoder_context_concat)

    # MODEL COMPILATION ============================================
    # Compile the Model with Gradient Clipping
    seq2seq_model_attention = Model([encoder_inputs, decoder_inputs], decoder_output)
    optimizer = RMSprop(clipvalue=1.0, learning_rate=0.01)  # Use gradient clipping to prevent large gradients
    seq2seq_model_attention.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # MODEL SUMMARY ============================================
    print(f"Seq2Seq Model with Attention Summary: {seq2seq_model_attention.summary()}")

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 Encoder_Input (InputLayer)  [(None, 30)]                 0         []                            
                                                                                                  
 Decoder_Input (InputLayer)  [(None, 30)]                 0         []                            
                                                                                                  
 encoder_embedding (Embeddi  (None, 30, 96)               6149049   ['Encoder_Input[0][0]']       
 ng)                                                      6                                       
                                                                                                  
 Decoder_Embedding (Embeddi  (None, 30, 96)               6149049   ['Decoder_Input[0][0]']   

In [ ]:
 # Initialize EarlyStopping to monitor the validation loss
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Initialize ReduceLROnPlateau to reduce the learning rate when the validation loss plateaus.
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

In [ ]:
# Train the model
print("Training Model With Attention:")
history_with_attention = seq2seq_model_attention.fit(
    [X_train_enc, X_train_dec], y_train,
    epochs=6,
    batch_size=256,
    validation_data=([X_val_enc, X_val_dec], y_val),
    callbacks=[early_stopping, lr_scheduler],
    verbose=1
)

Training Model With Attention:
Epoch 1/6
36922/36922 [==============================] - 7497s 200ms/step - loss: 4.7738 - accuracy: 0.2207 - val_loss: 4.6839 - val_accuracy: 0.2128 - lr: 0.0100
Epoch 2/6
36922/36922 [==============================] - 7138s 193ms/step - loss: 4.5498 - accuracy: 0.2337 - val_loss: 4.6598 - val_accuracy: 0.2109 - lr: 0.0100
Epoch 3/6
36922/36922 [==============================] - 7139s 193ms/step - loss: 4.5005 - accuracy: 0.2360 - val_loss: 4.5982 - val_accuracy: 0.2229 - lr: 0.0100
Epoch 4/6
36922/36922 [==============================] - 7137s 193ms/step - loss: 4.4634 - accuracy: 0.2379 - val_loss: 4.6127 - val_accuracy: 0.2234 - lr: 0.0100
Epoch 5/6
36922/36922 [==============================] - 7139s 193ms/step - loss: 4.4317 - accuracy: 0.2399 - val_loss: 4.5974 - val_accuracy: 0.2242 - lr: 0.0100
Epoch 6/6
36922/36922 [==============================] - 7138s 193ms/step - loss: 4.4061 - accuracy: 0.2418 - val_loss: 4.6028 - val_accuracy: 0.2245 - lr

In [ ]:
# Save the final Seq2Seq model with Attention
seq2seq_model_attention_path = os.path.join(config.MAIN_DIR_PATH, 'seq2seq_with_attention_model.h5')
seq2seq_model_attention.save(seq2seq_model_attention_path)
print(f"Model with attention saved successfully at {seq2seq_model_attention_path}")

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model with attention saved successfully at /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/seq2seq_model_with_attention2.h5
